# 🌿 Mental Health Support Chatbot – Fine-Tuned with Empathetic Dialogues

This notebook implements a **fine-tuned causal language model** (GPT‑2 based) capable of responding empathetically to user statements about emotions, stress, loneliness, or daily struggles.

## 🎯 Project Goal

To build a safe, supportive conversational agent that:
- Listens without judgment.
- Provides gentle, compassionate replies.
- Avoids harmful or insensitive outputs.

## 🧠 Methodology

1. **Dataset**: [Empathetic Dialogues](https://github.com/facebookresearch/EmpatheticDialogues) – 76k multi‑turn conversations labeled with emotions.
2. **Model**: `distilgpt2` (lightweight, efficient for fine‑tuning on a single GPU).
3. **Training**: Supervised fine‑tuning using a prompt template `Context: ... \nUser: ... \nResponse:` + next‑token prediction (causal LM).
4. **Safety**: Post‑generation filtering of unsafe phrases.
5. **Interface**: Gradio chat interface with example inputs.

## ⚠️ Important Note on Training Data

Training a full 76k‑sample model would take **several hours** (or days) on a T4 GPU. For quick experimentation and validation, **I use a random subset of 8,000 samples** (≈10% of the dataset).
The code for full‑data training is **commented out** – uncomment it if you have more time/compute resources.

## 📦 Dependencies

- `transformers`, `datasets`, `accelerate` – for model loading & training
- `gradio` – chat UI
- `torch` – GPU acceleration
- `wget`, `tarfile` – dataset download

##Environment Setup & Library Installation

In [1]:
# -*- coding: utf-8 -*-
# ============================================================
# TASK 5: Mental Health Support Chatbot (Fine-Tuned)
# ============================================================
!pip install -q transformers accelerate gradio datasets scikit-learn wget

import pandas as pd
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
import gradio as gr
import numpy as np

print("✅ All required libraries are ready.")

# Check for GPU to speed up training
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"✅ GPU is ready! You are using: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("⚠️ GPU is not available. Training will be slower on CPU.")

  Preparing metadata (setup.py) ... done
✅ All required libraries are ready.
✅ GPU is ready! You are using: Tesla T4


##Dataset Download & Loading (Empathetic Dialogues)

In [2]:
import tensorflow_datasets as tfds
import wget
import tarfile
import os
import pandas as pd

# Define the URL for the dataset
url = 'https://dl.fbaipublicfiles.com/parlai/empatheticdialogues/empatheticdialogues.tar.gz'

# Corrected path based on directory listing
train_file_relative_path = 'empatheticdialogues/train.csv'

# Create a directory for the dataset if it doesn't exist
output_dir = './empatheticdialogues_data'
os.makedirs(output_dir, exist_ok=True)

# Download the dataset tarball
print(f"Downloading dataset from {url}...")
archive_path = os.path.join(output_dir, 'empatheticdialogues.tar.gz')
if not os.path.exists(archive_path):
    wget.download(url, out=archive_path)
    print("\nDownload complete.")
else:
    print("\nArchive already exists. Skipping download.")

# Extract the dataset
print(f"Extracting dataset to {output_dir}...")
with tarfile.open(archive_path, 'r:gz') as tar:
    tar.extractall(path=output_dir)
print("Extraction complete.\n")

# --- List contents of the extracted directory (for verification) ---
print(f"Contents of the extracted directory '{output_dir}':")
for root, dirs, files in os.walk(output_dir):
    level = root.replace(output_dir, '').count(os.sep)
    indent = ' ' * 4 * (level)
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 4 * (level + 1)
    for f in files:
        print(f'{subindent}{f}')
# ---------------------------------------------------------------

# Load the train.csv file directly into a pandas DataFrame
train_file_path = os.path.join(output_dir, train_file_relative_path)

if os.path.exists(train_file_path):
    print(f"\n✅ Empathetic Dialogues training dataset found at: {train_file_path}")
    # Reverting to comma separator and skipping bad lines to handle ParserError
    df_train = pd.read_csv(train_file_path, sep=',', on_bad_lines='skip')

    # The 'prepare_data' function expects `context.numpy().decode()` and `utterance.numpy().decode()`.
    # Since we are loading from CSV, these are already strings. We need to mock the numpy() and decode() parts.
    class MockTensor:
        def __init__(self, value):
            self._value = value.encode('utf-8') if isinstance(value, str) else str(value).encode('utf-8')
        def numpy(self):
            return self._value

    # Initialize ds list
    ds = []
    # Group by conversation ID to reconstruct context-utterance pairs
    # Check if 'conv_id' and 'utterance_idx' columns exist after parsing
    if 'conv_id' in df_train.columns and 'utterance' in df_train.columns and 'utterance_idx' in df_train.columns:
        for conv_id, conversation_df in df_train.groupby('conv_id'):
            # Sort utterances by utterance_idx to maintain conversational order
            conversation_df = conversation_df.sort_values(by='utterance_idx')

            current_context = ""
            for index, row in conversation_df.iterrows():
                utterance = str(row['utterance']) # Ensure utterance is string
                # Add the current (context, utterance) pair to ds
                ds.append((MockTensor(current_context), MockTensor(utterance)))
                # Update the context for the next turn
                current_context += utterance + " "

        print(f"✅ Successfully loaded {len(ds)} context-utterance pairs into 'ds'.")
    else:
        print(f"⚠️ Required columns ('conv_id', 'utterance', 'utterance_idx') not found in the CSV after parsing.")
        print(f"Available columns: {df_train.columns.tolist()}")
        ds = [] # Ensure ds is defined as empty list

else:
    print(f"\n⚠️ Error: Training dataset not found at expected path: {train_file_path}")
    ds = [] # Ensure ds is defined even if file is not found

# The 'ds' variable is now populated correctly for the next cell's 'prepare_data' function.


Download complete.
Extracting dataset to ./empatheticdialogues_data...


/tmp/ipykernel_1117/1837816564.py:29: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=output_dir)


Extraction complete.

Contents of the extracted directory './empatheticdialogues_data':
empatheticdialogues_data/
    empatheticdialogues.tar.gz
    empatheticdialogues/
        test.csv
        train.csv
        valid.csv

✅ Empathetic Dialogues training dataset found at: ./empatheticdialogues_data/empatheticdialogues/train.csv
✅ Successfully loaded 76668 context-utterance pairs into 'ds'.


# Convert to DataFrame & Inspect Samples

In [3]:
data = []
for ctx, utt in ds:
  # Decode the mock tensor to a real string
    ctx_text = ctx.numpy().decode('utf-8') if hasattr(ctx, 'numpy') else str(ctx)
    utt_text = utt.numpy().decode('utf-8') if hasattr(utt, 'numpy') else str(utt)
    data.append({"context": ctx_text, "utterance": utt_text})

df = pd.DataFrame(data)
print(f"✅ DataFrame shape: {df.shape}")
print("\n📝 First 3 samples:")
print(df.head(3))

✅ DataFrame shape: (76668, 2)

📝 First 3 samples:
                                             context  \
0                                                      
1  I remember going to see the fireworks with my ...   
2  I remember going to see the fireworks with my ...   

                                           utterance  
0  I remember going to see the fireworks with my ...  
1  Was this a friend you were in love with_comma_...  
2                This was a best friend. I miss her.  


##Tokenization Setup (Context + Response Format)

In [8]:
print("\n🛠 Tokenizing with context and longer sequences...")

# Use distilgpt2 for speed; uncomment the next line to try the larger gpt2
#model_checkpoint = "gpt2"
model_checkpoint = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
tokenizer.pad_token = tokenizer.eos_token   # GPT-2 has no pad token by default

def preprocess_function(examples):
    """
    Build prompt: "Context: [situation]\nUser: [user utterance]\nResponse:"
    and the target is the utterance itself + EOS token.
    """
    inputs = [f"Context: {ctx}\nUser: {utt}\nResponse:"
              for ctx, utt in zip(examples["context"], examples["utterance"])]
    targets = [f"{utt}{tokenizer.eos_token}" for utt in examples["utterance"]]

    model_inputs = tokenizer(
        inputs,
        text_target=targets,
        max_length=256,          # increased from default (allows longer context)
        truncation=True,
        padding='max_length',
    )
    return model_inputs


🛠 Tokenizing with context and longer sequences...


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

##Data Preparation – Subsampling & Train/Test Split
 🚀 NOTE: Full‑data training is commented out below – uncomment only if you have enough time/GPU resources.

In [9]:
# ------------------------------------------------------------
# NOTE: The code below that uses the FULL dataset is COMMENTED OUT
# because training on all 76k samples would take several hours.
# Instead we take a random subset of 8000 samples to evaluate
# model quality and pipeline efficiency quickly.
# ------------------------------------------------------------

# ---------- FULL DATASET VERSION (COMMENTED OUT) ----------
#hf_dataset = Dataset.from_pandas(df)   # use all rows
#tokenized_dataset = hf_dataset.map(preprocess_function, batched=True, remove_columns=hf_dataset.column_names)

#split_dataset = tokenized_dataset.train_test_split(test_size=0.1, seed=42)
#train_dataset = split_dataset["train"]
#eval_dataset = split_dataset["test"]

#print(f"✅ Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

✅ Train: 18000, Eval: 2000


In [11]:
hf_dataset = Dataset.from_pandas(df)

total_samples = 8000   # 👈 change this number to use more/fewer samples

hf_dataset = hf_dataset.shuffle(seed=42).select(range(total_samples))
tokenized_dataset = hf_dataset.map(preprocess_function, batched=True, remove_columns=hf_dataset.column_names)

split_dataset = tokenized_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print(f"✅ Train samples: {len(train_dataset)}, Eval samples: {len(eval_dataset)}")

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

✅ Train samples: 7200, Eval samples: 800


##Model Loading & Training Configuration

In [12]:
model = AutoModelForCausalLM.from_pretrained(model_checkpoint)

training_args = TrainingArguments(
    output_dir="./mental-health-chatbot-improved",
    learning_rate=5e-5,
    per_device_train_batch_size=4,    # reduce to 2 if you encounter OOM errors
    per_device_eval_batch_size=4,
    num_train_epochs=10,              # increased from default
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    warmup_steps=500,                 # helps stabilize early training
    lr_scheduler_type="cosine",       # better than linear decay
    fp16=True,                        # enables mixed precision (faster on GPU)
    report_to="none",
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)
print("✅ Trainer ready.")

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

✅ Trainer ready.


##Fine‑Tuning (10 Epochs)
⏱️ This cell trains the model on the 8k subset. Expect ~30–40 minutes on T4 GPU.

In [13]:
print("\n⚙️ Starting Improved Fine-Tuning (10 epochs)...")
trainer.train()
print("✅ Fine-Tuning complete!")

# Save the final model and tokenizer
model_dir = "./finetuned-mental-health-bot-improved"
model.save_pretrained(model_dir)
tokenizer.save_pretrained(model_dir)
print(f"💾 Model saved to {model_dir}")


⚙️ Starting Improved Fine-Tuning (10 epochs)...


Epoch,Training Loss,Validation Loss
1,2.693669,2.631210
2,2.538727,2.566830
3,2.342896,2.534476
4,2.215846,2.537808
5,2.084702,2.546371
6,1.918947,2.569692
7,1.901039,2.592713
8,1.883726,2.603278
9,1.844497,2.620613
10,1.791673,2.626373


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Fine-Tuning complete!


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

💾 Model saved to ./finetuned-mental-health-bot-improved


##Build Generation Pipeline & Safety Filter

In [14]:
from transformers import pipeline

# Load the fine-tuned model from disk
chatbot_pipeline = pipeline("text-generation",
                            model=model_dir,
                            tokenizer=tokenizer,
                            device=0 if torch.cuda.is_available() else -1)

def is_safe_response(text):
  """Block responses that contain harmful phrases."""
    unsafe = ["kill yourself", "suicide", "self-harm", "worthless", "end your life", "die"]
    return not any(w in text.lower() for w in unsafe)

def generate_response(user_input, history):
    """
    Generate a response using the fine-tuned model.
    Uses the same prompt format as training: Context + User + Response:
    """
    prompt = f"Context: A user is feeling lonely or stressed.\nUser: {user_input}\nResponse:"

    try:
        response = chatbot_pipeline(
            prompt,
            max_new_tokens=60,          # limit response length
            do_sample=True,
            temperature=0.6,            # lower = more focused, higher = more creative
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )[0]['generated_text']

        # Extract only the part after "Response:"
        if "Response:" in response:
            response = response.split("Response:")[-1].strip()
        else:
            response = response.replace(prompt, "").strip()

        # Apply safety filter
        if not is_safe_response(response):
            response = "I care about you. Let's talk about something that might help you feel better."
    except Exception as e:
        response = "I'm here to listen. Could you tell me more?"
        print(f"Error: {e}")

    if not response:
        response = "Thank you for sharing. I'm here for you."

    return response

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

##Launch Gradio Chat Interface

In [15]:
demo = gr.ChatInterface(
    fn=generate_response,
    title="🌿 Mental Health Support Chatbot (Improved)",
    description="I'm here to listen and offer gentle, empathetic support. I'm not a therapist, but I care about you.",
    theme="soft",
    examples=[
        ["I feel so lonely today."],
        ["I'm really anxious about my job interview tomorrow."],
        ["I've been feeling sad for weeks."],
        ["I'm happy because I got a promotion!"]
    ]
)

demo.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://94b290769191fad8f0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
